## 환경 준비

아래 셀은 이 노트북에 필요한 Python 패키지가 설치되어 있는지 확인하고,
없으면 자동으로 설치한다. 이미 설치되어 있으면 빠르게 스킵된다.
터미널에서 미리 `uv sync`를 했다면 이 셀은 아무것도 설치하지 않는다.


In [ ]:
# === 의존성 자동 설치 (이미 설치되어 있으면 빠르게 스킵됨) ===
import subprocess, sys

_IMPORT_NAME_OVERRIDES = {
    "scikit-learn": "sklearn",
    "python-dateutil": "dateutil",
    "beautifulsoup4": "bs4",
}


def _ensure_packages(*packages):
    """누락된 패키지만 설치. 이미 있으면 스킵."""
    missing = []
    for pkg in packages:
        name = pkg.split(">=")[0].split("==")[0].split("[")[0]
        import_name = _IMPORT_NAME_OVERRIDES.get(name, name.replace("-", "_"))
        try:
            __import__(import_name)
        except ImportError:
            missing.append(pkg)
    if missing:
        print(f"Installing: {missing}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    else:
        print("All packages already installed ✓")

_ensure_packages(
    "pandas", "numpy", "matplotlib", "scikit-learn",
    "pydantic", "python-dateutil", "rich", "tqdm",
    "requests",  # OllamaClient에서 사용
)

# Ollama 서버 연결 확인
import requests
try:
    r = requests.get("http://localhost:11434/api/tags", timeout=5)
    models = [m["name"] for m in r.json().get("models", [])]
    print(f"Ollama connected ✓ models: {models}")
except Exception:
    print("⚠️ Ollama not available — LLM 기능은 규칙 기반으로 폴백됩니다")


# LLM 통합 — 규칙 기반에서 LLM 기반으로

이 노트북은 기존 규칙 기반 synthesizer/verifier 위에 Ollama 기반 LLM 경로를 덧붙이는 과정을 실습한다. 현재 시스템은 규칙 기반일 때 재현성과 설명 가능성이 높지만, 답변 표현력과 유연성은 제한적이다. 반대로 LLM을 붙이면 더 자연스럽고 압축된 답변을 만들 수 있지만, hallucination 위험과 외부 서버 의존성이 생긴다.

## 학습 목표
- `OllamaClient`가 어떤 payload로 로컬 모델과 통신하는지 이해한다.
- 규칙 기반 vs LLM 기반 synthesizer/verifier의 차이를 코드 수준에서 설명할 수 있다.
- `use_llm=True`가 workflow 안에서 어디를 바꾸고, 어디는 그대로 유지하는지 이해한다.
- LLM 서버가 꺼져도 폴백(fallback)으로 안전하게 동작하는 이유를 설명할 수 있다.


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import RuntimeConfig

runtime_config = RuntimeConfig.auto_detect()
print(sys.executable)
{
    "device": runtime_config.device,
    "llm_model": runtime_config.llm_model,
    "llm_base_url": runtime_config.llm_base_url,
    "llm_available": runtime_config.llm_available,
}

## 구현: OllamaClient 이해하기

**목적**
        - 로컬 Ollama 서버와 통신하는 최소 공용 클라이언트가 어떻게 생겼는지 먼저 이해한다.
        - 이후 셀에서 나오는 `chat()`, `chat_with_metadata()`, `is_available()` 호출이 어떤 내부 동작을 하는지 미리 잡아둔다.

        **핵심 로직**
        - `LLMConfig`는 base URL, 모델명, temperature, 최대 토큰 수, timeout을 묶은 설정 객체다.
        - `OllamaClient`는 `requests.Session`을 만들고, `/api/tags`와 `/api/chat`에 HTTP 요청을 보낸다.
        - `session.trust_env = False`로 프록시 환경변수 상속을 끊어, 로컬 Ollama 요청이 의도치 않게 외부 프록시를 타지 않게 막는다.

        **주요 파라미터**
        - `base_url`: 기본값은 `http://localhost:11434`이며, DGX Spark의 로컬 Ollama를 가리킨다.
        - `model`: 기본값은 `qwen3.5:9b`다. 이후 10번 notebook에서는 이 값을 `4b`, `2b`로 바꿔 비교한다.
        - `temperature`: 응답 다양성 조절값이다. 여기서는 grounded answer를 원하므로 낮게 둔다.
        - `max_tokens`: Ollama의 `num_predict`로 전달된다.
        - `timeout`: 느린 모델 호출이나 서버 지연 시 무한 대기를 막는다.

        **실제 소스 코드: src/llm_client.py 전체**
        ```python
        from __future__ import annotations

"""Ollama LLM client. OpenAI-compatible chat wrapper for local models."""

import os
from dataclasses import dataclass
from typing import Any
from urllib.parse import urlparse

import requests


@dataclass(frozen=True)
class LLMConfig:
    base_url: str = "http://localhost:11434"
    model: str = "qwen3.5:9b"
    temperature: float = 0.1
    max_tokens: int = 2048
    timeout: int = 120

    @classmethod
    def from_env(cls) -> "LLMConfig":
        return cls(
            base_url=os.getenv("OLLAMA_BASE_URL", "http://localhost:11434"),
            model=os.getenv("LLM_MODEL", "qwen3.5:9b"),
        )


class OllamaClient:
    def __init__(self, config: LLMConfig | None = None):
        self.config = config or LLMConfig()
        self.session = requests.Session()
        # Local Ollama traffic should not silently inherit proxy settings.
        self.session.trust_env = False

    @staticmethod
    def is_local_base_url(base_url: str) -> bool:
        parsed = urlparse(base_url)
        host = (parsed.hostname or "").lower()
        return host in {"localhost", "127.0.0.1", "::1"}

    def is_available(self) -> bool:
        try:
            response = self.session.get(f"{self.config.base_url}/api/tags", timeout=5)
            return response.status_code == 200
        except Exception:
            return False

    def list_models(self) -> list[str]:
        response = self.session.get(f"{self.config.base_url}/api/tags", timeout=5)
        response.raise_for_status()
        return [model["name"] for model in response.json().get("models", [])]

    def chat(self, messages: list[dict[str, str]], **kwargs: Any) -> str:
        payload = {
            "model": kwargs.get("model", self.config.model),
            "messages": messages,
            "stream": False,
            "options": {
                "temperature": kwargs.get("temperature", self.config.temperature),
                "num_predict": kwargs.get("max_tokens", self.config.max_tokens),
            },
        }
        response = self.session.post(
            f"{self.config.base_url}/api/chat",
            json=payload,
            timeout=kwargs.get("timeout", self.config.timeout),
        )
        response.raise_for_status()
        return response.json()["message"]["content"]

    def chat_with_metadata(self, messages: list[dict[str, str]], **kwargs: Any) -> dict[str, Any]:
        payload = {
            "model": kwargs.get("model", self.config.model),
            "messages": messages,
            "stream": False,
            "options": {
                "temperature": kwargs.get("temperature", self.config.temperature),
                "num_predict": kwargs.get("max_tokens", self.config.max_tokens),
            },
        }
        response = self.session.post(
            f"{self.config.base_url}/api/chat",
            json=payload,
            timeout=kwargs.get("timeout", self.config.timeout),
        )
        response.raise_for_status()
        data = response.json()
        return {
            "content": data["message"]["content"],
            "model": data.get("model", self.config.model),
            "eval_count": data.get("eval_count", 0),
            "eval_duration_ms": data.get("eval_duration", 0) / 1_000_000,
            "total_duration_ms": data.get("total_duration", 0) / 1_000_000,
        }
        ```

        **코드 읽기 포인트**
        - `LLMConfig.from_env()`는 환경변수로 base URL과 모델명을 덮어쓸 수 있어, notebook 코드를 바꾸지 않고 DGX 환경을 바꿀 수 있게 한다.
        - `is_available()`는 health check 역할을 한다. 노트북이 죽지 않고 폴백으로 넘어가려면 이 체크가 중요하다.
        - `chat()`는 assistant 텍스트만 반환하고, `chat_with_metadata()`는 `eval_count`, `eval_duration_ms`, `total_duration_ms`까지 반환해 성능 분석에 쓸 수 있게 한다.
        - payload 안 `options.num_predict`가 실제 generation length 제어값이다.

        **결과 해석 가이드**
        - `llm_live=True`면 실제 Ollama 서버와 연결된 것이다.
        - `installed_models`에 `qwen3.5:*`가 보이면 이후 비교 notebook에서 바로 쓸 수 있다.
        - `sample_metadata.eval_duration_ms`가 높으면 모델이 응답을 만드는 순수 생성 비용이 높다는 뜻이다.

        **💡 면접 포인트**
        - 로컬 LLM 연동에서도 단순 API 호출보다 health check와 proxy 차단이 운영 안정성에 중요하다.
        - metadata를 같이 수집하면 "좋아 보이는 답"뿐 아니라 "얼마나 비싼 답인지"까지 설명할 수 있다.

### src 코드 펼침: `OllamaClient.chat()`

```python
def chat(self, messages: list[dict[str, str]], **kwargs: Any) -> str:
    payload = {
        "model": kwargs.get("model", self.config.model),
        "messages": messages,
        "stream": False,
        "options": {
            "temperature": kwargs.get("temperature", self.config.temperature),
            "num_predict": kwargs.get("max_tokens", self.config.max_tokens),
        },
    }
    response = self.session.post(
        f"{self.config.base_url}/api/chat",
        json=payload,
        timeout=kwargs.get("timeout", self.config.timeout),
    )
    response.raise_for_status()
    return response.json()["message"]["content"]
```

- `kwargs.get("model", self.config.model)`은 호출 시점에 모델을 덮어쓸 수 있게 만든다. 같은 `OllamaClient` 인스턴스를 유지한 채 `qwen3.5:9b`, `4b`, `2b`를 번갈아 시험할 수 있다는 뜻이다.
- `messages`는 OpenAI 스타일 role/content 구조를 그대로 사용한다. Ollama도 `/api/chat`에서 이 구조를 받기 때문에 별도 SDK 없이 공통 인터페이스를 유지할 수 있다.
- `stream=False`는 토큰을 실시간으로 흘려보내지 않고 완성된 응답 전체를 한 번에 받겠다는 의미다. 교육용 notebook에서는 스트리밍보다 결과 비교와 파싱이 중요하므로 이 설정이 더 단순하다.
- `options.temperature`, `options.num_predict`는 생성 다양성과 최대 생성 길이를 제어한다. 여기서는 재현성을 위해 temperature를 낮게 두고, verifier JSON이 잘리지 않도록 충분한 길이를 준다.
- `timeout=...`은 서버가 느리거나 멈췄을 때 notebook 전체가 영원히 대기하지 않게 막는 안전장치다. 로컬 LLM 실험에서는 이 값이 없으면 체감상 "죽은 셀"처럼 보이기 쉽다.
- `response.raise_for_status()`는 HTTP 500, 404 같은 오류를 조용히 삼키지 않고 바로 예외로 올린다. 이 예외가 나중에 workflow 폴백을 타게 만드는 출발점이다.
- `response.json()["message"]["content"]`는 Ollama 응답 JSON에서 실제 assistant 텍스트만 뽑아낸다. notebook 코드에서는 결국 이 문자열 하나만 있으면 되므로 반환 타입도 단순하게 유지한다.

왜 OpenAI SDK가 아니라 `requests`를 직접 쓸까? 이 프로젝트는 Ollama의 로컬 HTTP API 구조를 그대로 이해하는 것이 목적이기 때문이다. 의존성을 최소화하면 설치가 단순해지고, payload와 응답 형식을 눈으로 따라가기 쉬워진다.

### src 코드 펼침: `OllamaClient.chat_with_metadata()`

```python
def chat_with_metadata(self, messages: list[dict[str, str]], **kwargs: Any) -> dict[str, Any]:
    payload = {
        "model": kwargs.get("model", self.config.model),
        "messages": messages,
        "stream": False,
        "options": {
            "temperature": kwargs.get("temperature", self.config.temperature),
            "num_predict": kwargs.get("max_tokens", self.config.max_tokens),
        },
    }
    response = self.session.post(
        f"{self.config.base_url}/api/chat",
        json=payload,
        timeout=kwargs.get("timeout", self.config.timeout),
    )
    response.raise_for_status()
    data = response.json()
    return {
        "content": data["message"]["content"],
        "model": data.get("model", self.config.model),
        "eval_count": data.get("eval_count", 0),
        "eval_duration_ms": data.get("eval_duration", 0) / 1_000_000,
        "total_duration_ms": data.get("total_duration", 0) / 1_000_000,
    }
```

- `eval_count`는 생성된 토큰 수에 가깝다. 같은 질문인데 9B가 더 긴 답을 내거나 2B가 짧게 끝내는 차이를 수치로 볼 수 있다.
- `eval_duration_ms`는 모델이 실제로 토큰을 생성하는 데 쓴 시간이다. 네트워크 오버헤드를 덜 반영하므로 모델 자체의 생성 속도를 읽을 때 유용하다.
- `total_duration_ms`는 요청 전체 왕복 시간이다. notebook에서 사용자가 체감하는 지연 시간은 이 값에 더 가깝다.
- 즉, `eval_duration_ms`는 "모델이 얼마나 빠른가", `total_duration_ms`는 "내가 실제로 얼마나 기다리는가"를 보여준다.


In [ ]:
from src.llm_client import LLMConfig, OllamaClient

llm_client = OllamaClient(LLMConfig.from_env())
llm_live = llm_client.is_available()

if llm_live:
    try:
        installed_models = llm_client.list_models()
        sample_response = llm_client.chat(
            [
                {"role": "system", "content": "Answer briefly."},
                {"role": "user", "content": "왜 grounding verification이 필요한가? 한 문장으로 설명해줘."},
            ]
        )
        sample_metadata = llm_client.chat_with_metadata(
            [
                {"role": "system", "content": "Answer briefly."},
                {"role": "user", "content": "Return one short sentence about evidence-grounded answers."},
            ]
        )
    except Exception as error:
        installed_models = []
        sample_response = f"Live call failed: {error}"
        sample_metadata = {"error": str(error)}
else:
    installed_models = []
    sample_response = "Ollama server unavailable. Later cells will demonstrate safe fallback."
    sample_metadata = {}

{
    "llm_live": llm_live,
    "configured_model": llm_client.config.model,
    "installed_models": installed_models[:5],
    "sample_response": sample_response,
    "sample_metadata": sample_metadata,
}

## 실험: 규칙 기반 vs LLM Synthesizer

**목적**
        - 같은 retrieved docs와 tool outputs를 넣었을 때, 규칙 기반 synthesizer와 LLM 기반 synthesizer가 어떻게 다른 답변을 만드는지 비교한다.

        **핵심 로직**
        - 먼저 baseline용 `synthesize_answer()`가 어떤 한계를 가지는지 보고,
        - 이어서 `synthesize_answer_llm()`가 evidence와 tool result를 어떤 프롬프트로 묶는지 본다.

        **주요 파라미터**
        - `query`: 사용자 질문이다.
        - `query_type`: summary, comparison, multi-hop 등 합성 전략을 바꾸는 신호다.
        - `retrieved_docs`: 답변의 근거 후보다.
        - `tool_outputs`: 계산값이나 날짜 파싱 결과처럼 문서 밖 보조 근거다.
        - `client`: Ollama 연결 객체다. 없거나 죽어 있으면 폴백한다.

        기존 규칙 기반 synthesizer는 핵심 문장을 재정렬한 뒤 이어 붙이는 방식이라 빠르고 안정적이지만, 문장 간 연결이 딱딱하고 citation 파싱도 정교하지 않다. 아래 새 모듈은 같은 입력을 LLM에게 넘기되, "제공된 evidence만 사용하고 `[source_name]` citation을 포함하라"는 제약을 프롬프트에 직접 넣는다.

        **실제 소스 코드: src/synthesizer_llm.py 전체**
        ```python
        from __future__ import annotations

import re
import warnings
from typing import Any

from src.synthesizer import synthesize_answer


def _format_evidence(retrieved_docs: list[dict[str, Any]]) -> str:
    if not retrieved_docs:
        return "No retrieved documents."

    lines: list[str] = []
    for index, doc in enumerate(retrieved_docs, start=1):
        lines.append(
            "\n".join(
                [
                    f"[{index}] source={doc['source']}",
                    f"doc_id={doc['doc_id']} chunk_id={doc['chunk_id']} score={doc.get('score', 0.0)}",
                    f"text={doc['text']}",
                ]
            )
        )
    return "\n\n".join(lines)


def _format_tools(tool_outputs: list[dict[str, Any]]) -> str:
    if not tool_outputs:
        return "No tool results."
    return "\n".join(f"- {output['tool_name']}: {output['output']}" for output in tool_outputs)


def _default_citations(retrieved_docs: list[dict[str, Any]]) -> list[dict[str, Any]]:
    return [
        {
            "doc_id": doc["doc_id"],
            "chunk_id": doc["chunk_id"],
            "source": doc["source"],
            "score": doc.get("score", 0.0),
        }
        for doc in retrieved_docs[:3]
    ]


def _parse_citations(answer: str, retrieved_docs: list[dict[str, Any]]) -> list[dict[str, Any]]:
    cited_sources = {match.strip() for match in re.findall(r"\[([^\[\]]+)\]", answer)}
    citations = [
        {
            "doc_id": doc["doc_id"],
            "chunk_id": doc["chunk_id"],
            "source": doc["source"],
            "score": doc.get("score", 0.0),
        }
        for doc in retrieved_docs
        if doc["source"] in cited_sources
    ]
    return citations or _default_citations(retrieved_docs)


def synthesize_answer_llm(
    query: str,
    query_type: str,
    retrieved_docs: list[dict[str, Any]],
    tool_outputs: list[dict[str, Any]],
    client: Any = None,
) -> dict[str, Any]:
    """Generate a grounded answer from retrieved evidence via an LLM, with safe fallback."""

    fallback = synthesize_answer(
        query=query,
        query_type=query_type,
        retrieved_docs=retrieved_docs,
        tool_outputs=tool_outputs,
    )

    if client is None:
        warnings.warn("LLM synthesizer fallback: no client provided.", stacklevel=2)
        return fallback
    if hasattr(client, "is_available") and not client.is_available():
        warnings.warn("LLM synthesizer fallback: client unavailable.", stacklevel=2)
        return fallback

    messages = [
        {
            "role": "system",
            "content": (
                "You are a research assistant. Answer ONLY based on the provided evidence. "
                "If insufficient, say so. Include [source_name] citations."
            ),
        },
        {
            "role": "user",
            "content": (
                f"Evidence:\n{_format_evidence(retrieved_docs)}\n\n"
                f"Tool results:\n{_format_tools(tool_outputs)}\n\n"
                f"Question: {query}"
            ),
        },
    ]

    try:
        draft_answer = client.chat(messages)
    except Exception as error:
        warnings.warn(f"LLM synthesizer fallback: {error}", stacklevel=2)
        return fallback

    return {
        "draft_answer": draft_answer,
        "citations": _parse_citations(draft_answer, retrieved_docs),
    }
        ```

        **코드 읽기 포인트**
        - `_format_evidence()`는 retrieved docs를 `[1] source=...` 형식으로 펼쳐 LLM이 근거를 직접 읽게 만든다.
        - `_format_tools()`는 calculator나 keyword extractor 결과를 프롬프트에 넣어, retrieval만으로 답하기 어려운 부분을 보조한다.
        - `fallback = synthesize_answer(...)`를 먼저 만들어 두는 것이 중요하다. LLM 경로가 실패해도 항상 규칙 기반 답변으로 복귀할 수 있다.
        - `_parse_citations()`는 LLM 응답에서 `[source_name]` 패턴을 찾아 citation을 구성한다. 파싱 실패 시 retrieved_docs 상위 3개로 안전하게 폴백한다.

        **결과 해석 가이드**
        - `draft_answer`가 더 자연스럽다고 해서 더 좋은 답이라는 뜻은 아니다. citation과 coverage를 함께 봐야 한다.
        - `llm_warning`이 있으면 실제 LLM이 아니라 규칙 기반 폴백이 사용됐을 가능성이 크다.
        - `citation_count`가 유지되거나 늘었는지 보면 LLM 답변이 근거를 얼마나 드러내는지 가늠할 수 있다.

        **💡 면접 포인트**
        - LLM synthesizer를 붙일 때 핵심은 "자연스러움"보다 "evidence-only" 제약과 안전한 fallback이다.
        - citation 파싱 실패를 대비해 기본 citation 경로를 남겨 두면 운영 안정성이 높아진다.

### src 코드 펼침: 규칙 기반 `synthesize_answer()`

```python
def synthesize_answer(
    query: str,
    query_type: str,
    retrieved_docs: list[dict[str, Any]],
    tool_outputs: list[dict[str, Any]],
) -> dict[str, Any]:
    normalized_query = normalize_text(query).lower()
    ranked = _rank_sentences(query, retrieved_docs)
    evidence_sentences = [sentence for _, sentence, _ in ranked[:3]]
    ...
    if not evidence_sentences:
        draft = "I could not find grounded evidence for this question in the loaded documents."
        return {"draft_answer": draft, "citations": citations}
    ...
    draft = f"{body} Sources: {source_list}."
    return {"draft_answer": draft, "citations": citations}
```

- 규칙 기반 synthesizer는 사실상 "점수가 높은 문장 몇 개를 뽑아 질문 유형별 템플릿에 맞게 이어 붙이는 로직"이다.
- `_rank_sentences()`가 retrieval 결과 안에서 문장 단위 재정렬을 수행하고, `query_type`에 따라 첫 문장 하나를 쓰거나 두 문장을 붙이거나 비교 문장을 우선 선택한다.
- 장점은 빠르고 결정적(deterministic)이라는 점이다. 같은 입력이면 같은 답이 나온다.
- 한계도 분명하다. 문장이 자연스럽게 연결되도록 다시 쓰지 못하고, 여러 문서를 묶어 새로운 표현으로 재구성하는 능력이 약하다. 그래서 LLM 기반 synthesizer가 필요한 이유가 생긴다.

### src 코드 펼침: `_parse_citations()`

```python
def _parse_citations(answer: str, retrieved_docs: list[dict[str, Any]]) -> list[dict[str, Any]]:
    cited_sources = {match.strip() for match in re.findall(r"\[([^\[\]]+)\]", answer)}
    citations = [
        {
            "doc_id": doc["doc_id"],
            "chunk_id": doc["chunk_id"],
            "source": doc["source"],
            "score": doc.get("score", 0.0),
        }
        for doc in retrieved_docs
        if doc["source"] in cited_sources
    ]
    return citations or _default_citations(retrieved_docs)
```

- LLM에게 `[source_name]` 형식으로 citation을 쓰라고 시키는 이유가 여기서 드러난다. 정규식 `r"\[([^\[\]]+)\]"`로 대괄호 안 문자열만 깔끔하게 뽑아낼 수 있기 때문이다.
- LLM이 citation을 아예 빼먹거나 엉뚱한 형식으로 쓰면 `citations`가 비게 된다. 이때 `_default_citations(retrieved_docs)`로 상위 문서 3개를 채워 넣어 trace와 UI가 완전히 비지 않게 만든다.
- 즉, citation 파싱은 "정확하면 더 좋고, 실패해도 시스템은 죽지 않는다"라는 방어적 설계(defensive design)다.

### src 코드 펼침: `synthesize_answer_llm()`

```python
def synthesize_answer_llm(
    query: str,
    query_type: str,
    retrieved_docs: list[dict[str, Any]],
    tool_outputs: list[dict[str, Any]],
    client: Any = None,
) -> dict[str, Any]:
    fallback = synthesize_answer(...)

    if client is None:
        warnings.warn("LLM synthesizer fallback: no client provided.", stacklevel=2)
        return fallback
    if hasattr(client, "is_available") and not client.is_available():
        warnings.warn("LLM synthesizer fallback: client unavailable.", stacklevel=2)
        return fallback

    messages = [
        {
            "role": "system",
            "content": (
                "You are a research assistant. Answer ONLY based on the provided evidence. "
                "If insufficient, say so. Include [source_name] citations."
            ),
        },
        {
            "role": "user",
            "content": (
                f"Evidence:
{_format_evidence(retrieved_docs)}

"
                f"Tool results:
{_format_tools(tool_outputs)}

"
                f"Question: {query}"
            ),
        },
    ]

    try:
        draft_answer = client.chat(messages)
    except Exception as error:
        warnings.warn(f"LLM synthesizer fallback: {error}", stacklevel=2)
        return fallback

    return {
        "draft_answer": draft_answer,
        "citations": _parse_citations(draft_answer, retrieved_docs),
    }
```

- 이 함수의 출발점은 항상 `fallback = synthesize_answer(...)`다. 즉, LLM 경로는 규칙 기반 로직 위에 "더 나은 표현과 재구성 능력"을 얹는 선택 레이어이고, 밑바닥 안전망은 이미 준비돼 있다.
- system 프롬프트의 핵심 문구는 `Answer ONLY based on the provided evidence.`다. 이 한 줄이 없으면 모델은 자기 사전지식(parametric memory)을 섞어 넣을 가능성이 커진다.
- `If insufficient, say so.`는 근거 부족 시 억지 답변을 만들지 말라는 지시다. verifier가 뒤에서 다시 잡아주긴 하지만, 생성 단계부터 보수적으로 유도하는 편이 더 안정적이다.
- `_format_evidence(retrieved_docs)`는 검색 결과를 `[1] source=...`, `doc_id=...`, `chunk_id=...`, `score=...`, `text=...` 순서로 번호 매겨 넣는다. LLM이 여러 chunk를 구분하고 인용 출처를 재사용하기 쉽게 만드는 포맷이다.
- `_format_tools(tool_outputs)`는 계산기나 날짜 파서 결과를 별도 블록으로 제공한다. 문서에는 없는 계산 결과를 답변에 반영해야 할 때 중요하다.
- 폴백 조건은 세 가지다. `client is None`, `client.is_available() == False`, `client.chat(...)` 중 예외 발생. 어느 경우든 경고만 남기고 규칙 기반 답변으로 복귀한다.

### 데이터 전환 포인트

이번 셀부터는 `build_demo_index()`로 29개 chunk만 보는 대신 `load_profile('demo')`와 `load_profile('tech_docs')`를 동시에 로드한다. 같은 synthesizer 비교라도 코퍼스 규모가 29개 chunk인지, 9천 개가 넘는 chunk인지에 따라 차이가 훨씬 크게 드러나기 때문이다.

읽을 때는 두 축을 같이 보자.
- `demo`: 문서가 짧고 구조가 단순해서 규칙 기반도 핵심 문장을 쉽게 뽑는다.
- `tech_docs`: 문서가 길고 개념이 중첩돼 있어 LLM이 여러 chunk를 압축·재구성하는 장점이 더 잘 보인다.

즉 여기서 중요한 것은 "LLM이 항상 더 좋다"가 아니라, 데이터 복잡도가 올라갈수록 LLM 도입 효과가 얼마나 커지는지다.


In [ ]:
import warnings

import pandas as pd
from IPython.display import display

from src.classifier import classify_query
from src.data_profiles import load_profile
from src.synthesizer import synthesize_answer
from src.synthesizer_llm import synthesize_answer_llm
from src.tools import execute_tool_requests, plan_tool_requests


def preview(text: str, limit: int = 120) -> str:
    compact = " ".join(text.split())
    return compact if len(compact) <= limit else compact[: limit - 3] + "..."


demo_profile = load_profile('demo', persist=False)
demo_retriever = demo_profile['retriever']

tech_profile = load_profile('tech_docs', persist=False)
tech_retriever = tech_profile['retriever']

print(f"Demo: {demo_profile['stats']}")
print(f"Tech docs: {tech_profile['stats']}")

demo_queries = {
    'simple_lookup': 'What are the goals of the workspace policy refresh?',
    'comparison': 'How is the rollout plan different from the policy refresh?',
    'multi_hop': 'How many days are in the pilot window?',
    'summary': 'Summarize the loaded documents.',
    'insufficient_evidence_risk': 'What is the current stock price?',
}

tech_queries = {
    'simple_lookup': 'How does Anthropic recommend structuring tool definitions?',
    'comparison': "What is the difference between LangGraph's Graph API and Functional API?",
    'multi_hop': 'How does sentence-transformers training loss affect embedding quality?',
    'summary': 'Summarize the key concepts of LangGraph.',
    'insufficient_evidence_risk': "What is OpenAI's current pricing for GPT-5?",
}

profiles = {
    'demo': {'profile': demo_profile, 'retriever': demo_retriever, 'queries': demo_queries},
    'tech_docs': {'profile': tech_profile, 'retriever': tech_retriever, 'queries': tech_queries},
}

display(
    pd.DataFrame(
        [
            {'dataset': name, **bundle['profile']['stats']}
            for name, bundle in profiles.items()
        ]
    )[['dataset', 'document_count', 'chunk_count', 'eval_question_count', 'avg_chunk_chars']]
)

focus_query_type = 'summary'
comparison_contexts = {}
synth_rows = []
warning_rows = []

for dataset_name, bundle in profiles.items():
    query = bundle['queries'][focus_query_type]
    classification = classify_query(query)
    retrieved_docs = bundle['retriever'].search(query, top_k=5)
    tool_requests = plan_tool_requests(query, classification['query_type'], retrieved_docs)
    tool_outputs = [output.to_dict() for output in execute_tool_requests(tool_requests)]

    rule_result = synthesize_answer(
        query=query,
        query_type=classification['query_type'],
        retrieved_docs=retrieved_docs,
        tool_outputs=tool_outputs,
    )

    with warnings.catch_warnings(record=True) as synth_warnings:
        warnings.simplefilter('always')
        llm_result = synthesize_answer_llm(
            query=query,
            query_type=classification['query_type'],
            retrieved_docs=retrieved_docs,
            tool_outputs=tool_outputs,
            client=llm_client,
        )

    comparison_contexts[dataset_name] = {
        'query': query,
        'classification': classification,
        'retrieved_docs': retrieved_docs,
        'tool_outputs': tool_outputs,
        'rule_result': rule_result,
        'llm_result': llm_result,
        'warning_messages': [str(item.message) for item in synth_warnings] or ['No warning'],
    }

    synth_rows.extend(
        [
            {
                'dataset': dataset_name,
                'chunk_count': bundle['profile']['stats']['chunk_count'],
                'query_type': classification['query_type'],
                'system': 'rule_based',
                'draft_answer': preview(rule_result['draft_answer'], limit=180),
                'citation_count': len(rule_result['citations']),
                'retrieved_doc_count': len(retrieved_docs),
            },
            {
                'dataset': dataset_name,
                'chunk_count': bundle['profile']['stats']['chunk_count'],
                'query_type': classification['query_type'],
                'system': 'llm_based',
                'draft_answer': preview(llm_result['draft_answer'], limit=180),
                'citation_count': len(llm_result['citations']),
                'retrieved_doc_count': len(retrieved_docs),
            },
        ]
    )
    warning_rows.append(
        {
            'dataset': dataset_name,
            'llm_mode': 'live' if llm_live and not synth_warnings else 'fallback',
            'llm_warning': ' | '.join(str(item.message) for item in synth_warnings) or 'No warning',
        }
    )

synth_comparison = pd.DataFrame(synth_rows)
display(synth_comparison)
pd.DataFrame(warning_rows)


## 결과 해석: 답변 품질과 hallucination 위험

LLM 답변이 더 짧고 자연스럽게 보이면 사용자는 보통 더 좋아한다. 하지만 그 자연스러움이 곧 grounded answer를 보장하지는 않는다. 오히려 표현력이 좋아질수록 문서에 없는 내용을 그럴듯하게 보충할 위험도 커진다. 그래서 synthesizer 비교 다음에 verifier 비교가 바로 붙어야 한다.

이 표를 읽을 때는 아래 기준을 같이 보자.
- 규칙 기반 답변은 문장 이어 붙이기라 덜 자연스럽지만, 어떤 근거 문장에서 왔는지 추적이 쉽다.
- LLM 기반 답변은 문장 압축과 연결이 자연스러울 수 있지만, 실제로는 문서 밖 내용을 섞었는지 verifier로 다시 확인해야 한다.
- 경고가 떴다면 품질 비교가 아니라 "폴백 설계가 제대로 작동했는가"를 보는 셀로 읽는 편이 맞다.

### 데이터 규모 차이로 읽는 법

같은 `summary` 질문이라도 `demo`에서는 retrieved docs 몇 개만 읽으면 답이 거의 정리된다. 반면 `tech_docs`는 하나의 개념이 여러 chunk와 여러 문서에 흩어져 있어, 문장 이어 붙이기만으로는 요약 품질 차이가 잘 드러난다.

그래서 이 표에서 `demo` 행은 "규칙 기반도 꽤 괜찮다"는 기준선으로 읽고, `tech_docs` 행은 "LLM이 왜 필요한가"를 보여주는 확장 실험으로 읽는 편이 맞다.

**💡 면접 포인트**
- 데이터 규모와 문서 복잡도가 낮으면 규칙 기반으로도 충분할 수 있다.
- 코퍼스가 커지고 chunk 간 연결이 중요해질수록 LLM의 합성 능력이 실제 품질 차이로 나타난다.


## 구현: 규칙 기반 vs LLM Verifier

**목적**
        - 답변이 근거에 기반하는지 판단하는 verifier를 규칙 기반과 LLM 기반 두 경로로 비교한다.

        **핵심 로직**
        - 규칙 기반 verifier는 토큰 겹침과 tool output 보정으로 `coverage_score`를 계산한다.
        - LLM 기반 verifier는 JSON 응답만 허용하는 프롬프트를 사용해 구조화된 판정을 요청한다.
        - JSON 파싱 실패나 연결 실패가 나면 즉시 규칙 기반 verifier로 돌아간다.

        **주요 파라미터**
        - `draft_answer`: 검증 대상 답변이다.
        - `retrieved_docs`: 근거 문서다.
        - `tool_outputs`: 계산값처럼 토큰 겹침이 낮아도 합법적인 근거를 보정하는 신호다.
        - `client`: LLM verifier 호출용 Ollama client다.

        **실제 소스 코드: src/verifier_llm.py 전체**
        ```python
        from __future__ import annotations

import json
import re
import warnings
from typing import Any

from src.schemas import VerificationResult
from src.verifier import verify_grounding


def _format_evidence(retrieved_docs: list[dict[str, Any]]) -> str:
    if not retrieved_docs:
        return "No retrieved documents."

    lines: list[str] = []
    for index, doc in enumerate(retrieved_docs, start=1):
        lines.append(
            "\n".join(
                [
                    f"[{index}] source={doc['source']}",
                    f"doc_id={doc['doc_id']} chunk_id={doc['chunk_id']} score={doc.get('score', 0.0)}",
                    f"text={doc['text']}",
                ]
            )
        )
    return "\n\n".join(lines)


def _extract_json_blob(response: str) -> dict[str, Any]:
    match = re.search(r"\{.*\}", response, re.DOTALL)
    if not match:
        raise ValueError("No JSON object found in verifier response.")
    return json.loads(match.group(0))


def verify_grounding_llm(
    query: str,
    draft_answer: str,
    retrieved_docs: list[dict[str, Any]],
    tool_outputs: list[dict[str, Any]] | None = None,
    client: Any = None,
) -> VerificationResult:
    """Verify grounding through an LLM JSON response, with safe fallback."""

    fallback = verify_grounding(
        query=query,
        draft_answer=draft_answer,
        retrieved_docs=retrieved_docs,
        tool_outputs=tool_outputs,
    )

    if client is None:
        warnings.warn("LLM verifier fallback: no client provided.", stacklevel=2)
        return fallback
    if hasattr(client, "is_available") and not client.is_available():
        warnings.warn("LLM verifier fallback: client unavailable.", stacklevel=2)
        return fallback

    messages = [
        {
            "role": "system",
            "content": (
                'You are a grounding verifier. Respond ONLY with JSON: '
                '{"is_grounded": bool, "coverage_score": float 0-1, '
                '"unsupported_claims": [...], "missing_aspects": [...]}'
            ),
        },
        {
            "role": "user",
            "content": (
                f"Evidence:\n{_format_evidence(retrieved_docs)}\n\n"
                f"Answer:\n{draft_answer}\n\n"
                f"Question: {query}"
            ),
        },
    ]

    try:
        raw_response = client.chat(messages)
        parsed = _extract_json_blob(raw_response)
        return VerificationResult(
            is_grounded=bool(parsed["is_grounded"]),
            coverage_score=round(float(parsed["coverage_score"]), 3),
            unsupported_claims=[str(item) for item in parsed.get("unsupported_claims", [])],
            missing_aspects=[str(item) for item in parsed.get("missing_aspects", [])],
        )
    except Exception as error:
        warnings.warn(f"LLM verifier fallback: {error}", stacklevel=2)
        return fallback
        ```

        **코드 읽기 포인트**
        - `fallback = verify_grounding(...)`를 먼저 계산해 두기 때문에, LLM verifier는 실패해도 시스템이 멈추지 않는다.
        - system prompt가 JSON만 반환하라고 강제하는 이유는 downstream parsing을 안정화하기 위해서다.
        - `_extract_json_blob()`는 응답 안 `Ellipsis` 블록만 정규식으로 뽑는다. 모델이 약간의 군더더기 텍스트를 붙여도 살릴 수 있다.
        - `VerificationResult`로 다시 감싸기 때문에 규칙 기반/LLM 기반 verifier가 같은 인터페이스를 유지한다.

        **결과 해석 가이드**
        - `is_grounded`는 최종 통과/실패 판정이고, `coverage_score`는 그 강도를 수치화한 값이다.
        - `unsupported_claims`가 비어 있지 않으면 답변 일부가 근거 밖으로 나갔다는 뜻이다.
        - `missing_aspects`는 질문 전체 중 아직 근거가 충분하지 않은 부분을 알려 주므로, retrieval 부족인지 synthesis 부족인지 추적할 때 유용하다.

        **💡 면접 포인트**
        - LLM verifier를 붙이더라도 구조화된 JSON 계약을 강제하지 않으면 운영에서 쓰기 어렵다.
        - verifier 실패 시 규칙 기반 검증으로 폴백하는 설계가 신뢰성과 가용성을 동시에 지켜 준다.

### src 코드 펼침: 규칙 기반 `verify_grounding()`

```python
def verify_grounding(
    query: str,
    draft_answer: str,
    retrieved_docs: list[dict[str, Any]],
    tool_outputs: list[dict[str, Any]] | None = None,
) -> VerificationResult:
    if not retrieved_docs:
        return VerificationResult(...)

    evidence_token_sets = [content_tokens(doc["text"]) for doc in retrieved_docs]
    answer_sentences = [
        sentence
        for sentence in sentence_split(draft_answer)
        if not sentence.lower().startswith("sources:")
    ]
    ...
    for sentence in answer_sentences:
        sentence_tokens = content_tokens(sentence)
        support = max(overlap_ratio(sentence_tokens, evidence_tokens) for evidence_tokens in evidence_token_sets)
        ...
        if support < 0.45:
            unsupported_claims.append(sentence)

    query_coverage = overlap_ratio(content_tokens(query), evidence_union)
    answer_coverage = sum(support_scores) / len(support_scores) if support_scores else 0.0
    coverage_score = round(min(1.0, (answer_coverage * 0.75) + (query_coverage * 0.25)), 3)
    is_grounded = not unsupported_claims and coverage_score >= 0.6
    return VerificationResult(...)
```

- 규칙 기반 verifier는 답변 문장을 다시 잘게 쪼개고, 각 문장이 검색 근거 토큰과 얼마나 겹치는지 계산한다.
- `support < 0.45`이면 unsupported claim으로 본다. 즉, 문장이 완전히 거짓이라고 단정하는 것이 아니라 "현재 evidence만으로는 뒷받침이 약하다"고 보는 것이다.
- 최종 `coverage_score`는 `answer_coverage * 0.75 + query_coverage * 0.25`다. 답변 문장 자체의 근거성에 더 큰 비중을 두고, 질문 전체를 얼마나 포괄했는지는 보조 지표로 쓴다.
- 이 방식은 빠르고 설명 가능하지만, 의미는 같아도 표현이 다르면 과하게 낮게 나올 수 있다. 그래서 LLM 기반 verifier를 추가로 실험하는 것이다.

### src 코드 펼침: `_extract_json_blob()`

```python
def _extract_json_blob(response: str) -> dict[str, Any]:
    match = re.search(r"\{.*\}", response, re.DOTALL)
    if not match:
        raise ValueError("No JSON object found in verifier response.")
    return json.loads(match.group(0))
```

- LLM은 `Respond ONLY with JSON`이라고 말해도 종종 앞뒤에 설명 문장을 붙인다. 그래서 응답 전체를 바로 `json.loads()` 하면 깨질 수 있다.
- 정규식으로 가장 큰 `{...}` 블록을 먼저 뽑아낸 뒤 그 부분만 `json.loads()` 하는 이유가 여기에 있다.
- 이 함수가 실패하면 곧바로 예외가 올라가고, 바깥의 `except` 블록이 규칙 기반 verifier로 폴백한다.

### src 코드 펼침: `verify_grounding_llm()`

```python
def verify_grounding_llm(
    query: str,
    draft_answer: str,
    retrieved_docs: list[dict[str, Any]],
    tool_outputs: list[dict[str, Any]] | None = None,
    client: Any = None,
) -> VerificationResult:
    fallback = verify_grounding(...)

    if client is None:
        warnings.warn("LLM verifier fallback: no client provided.", stacklevel=2)
        return fallback
    if hasattr(client, "is_available") and not client.is_available():
        warnings.warn("LLM verifier fallback: client unavailable.", stacklevel=2)
        return fallback

    messages = [
        {
            "role": "system",
            "content": (
                'You are a grounding verifier. Respond ONLY with JSON: '
                '{"is_grounded": bool, "coverage_score": float 0-1, '
                '"unsupported_claims": [...], "missing_aspects": [...]}'
            ),
        },
        {
            "role": "user",
            "content": (
                f"Evidence:
{_format_evidence(retrieved_docs)}

"
                f"Answer:
{draft_answer}

"
                f"Question: {query}"
            ),
        },
    ]

    try:
        raw_response = client.chat(messages)
        parsed = _extract_json_blob(raw_response)
        return VerificationResult(
            is_grounded=bool(parsed["is_grounded"]),
            coverage_score=round(float(parsed["coverage_score"]), 3),
            unsupported_claims=[str(item) for item in parsed.get("unsupported_claims", [])],
            missing_aspects=[str(item) for item in parsed.get("missing_aspects", [])],
        )
    except Exception as error:
        warnings.warn(f"LLM verifier fallback: {error}", stacklevel=2)
        return fallback
```

- system 프롬프트가 `Respond ONLY with JSON`을 강하게 요구하는 이유는 후처리 비용을 줄이기 위해서다. verifier는 사람이 읽는 문장보다 구조화된 판정 결과가 더 중요하다.
- `coverage_score`를 0~1 부동소수점으로 제한한 것도 중요하다. 나중에 fallback threshold와 직접 비교할 수 있기 때문이다.
- `VerificationResult(...)`로 바로 매핑하면 이후 workflow의 `fallback_or_finalize()`는 규칙 기반이든 LLM 기반이든 같은 자료형을 받는다. 즉, 상위 레이어는 verifier 내부 구현을 몰라도 된다.
- JSON 파싱 실패, 필드 누락, 서버 오류, timeout은 모두 `except`로 모아 규칙 기반 verifier로 되돌린다. 실무에서 가장 흔한 LLM 장애 대응 패턴이다.


In [ ]:
import warnings

import pandas as pd
from IPython.display import display

from src.verifier import verify_grounding
from src.verifier_llm import verify_grounding_llm

verifier_rows = []
verifier_warning_rows = []

for dataset_name, context in comparison_contexts.items():
    draft_answer = context['llm_result']['draft_answer']
    rule_verification = verify_grounding(
        query=context['query'],
        draft_answer=draft_answer,
        retrieved_docs=context['retrieved_docs'],
        tool_outputs=context['tool_outputs'],
    )

    with warnings.catch_warnings(record=True) as verifier_warnings:
        warnings.simplefilter('always')
        llm_verification = verify_grounding_llm(
            query=context['query'],
            draft_answer=draft_answer,
            retrieved_docs=context['retrieved_docs'],
            tool_outputs=context['tool_outputs'],
            client=llm_client,
        )

    verifier_rows.extend(
        [
            {
                'dataset': dataset_name,
                'verifier': 'rule_based',
                'is_grounded': rule_verification.is_grounded,
                'coverage_score': rule_verification.coverage_score,
                'unsupported_claims': ' | '.join(rule_verification.unsupported_claims) or 'none',
                'missing_aspects': ' | '.join(rule_verification.missing_aspects) or 'none',
            },
            {
                'dataset': dataset_name,
                'verifier': 'llm_based',
                'is_grounded': llm_verification.is_grounded,
                'coverage_score': llm_verification.coverage_score,
                'unsupported_claims': ' | '.join(llm_verification.unsupported_claims) or 'none',
                'missing_aspects': ' | '.join(llm_verification.missing_aspects) or 'none',
            },
        ]
    )
    verifier_warning_rows.append(
        {
            'dataset': dataset_name,
            'warning_count': len(verifier_warnings),
            'warnings': ' | '.join(str(item.message) for item in verifier_warnings) or 'No warning',
        }
    )

display(pd.DataFrame(verifier_rows))
pd.DataFrame(verifier_warning_rows)


## 구현: 통합 워크플로우 (`use_llm=True`)

**목적**
        - LLM 통합이 workflow 전체를 갈아엎는 것이 아니라, synthesis와 verification 두 노드만 선택적으로 바꾼다는 점을 확인한다.

        **핵심 로직**
        - `run_workflow(..., use_llm=True, llm_client=...)`일 때만 `synthesize_answer_llm()`와 `verify_grounding_llm()`를 탄다.
        - 나머지 normalize, classify, plan, retrieve, tool, fallback 구조는 동일하다.
        - 즉 observability와 state contract는 그대로 유지한 채 generation/verification 레이어만 바꾸는 구조다.

        **주요 파라미터**
        - `use_llm`: LLM 경로를 켜는 스위치다.
        - `llm_client`: 명시적으로 주입할 Ollama client다. 없으면 내부에서 기본 client를 만들 수 있다.
        - `query_suite`: 5가지 query type별 비교용 질문 묶음이다.

        **실제 소스 코드: workflow.py의 LLM 분기 핵심**
        ```python
        def synthesize_answer_node(
    state: AgentState,
    use_llm: bool = False,
    llm_client: OllamaClient | None = None,
) -> None:
    start = time.perf_counter()
    if use_llm:
        result = synthesize_answer_llm(
            query=state["normalized_query"],
            query_type=state["query_type"],
            retrieved_docs=state["retrieved_docs"],
            tool_outputs=state["tool_outputs"],
            client=llm_client,
        )
    else:
        result = synthesize_answer(
            query=state["normalized_query"],
            query_type=state["query_type"],
            retrieved_docs=state["retrieved_docs"],
            tool_outputs=state["tool_outputs"],
        )
    state["draft_answer"] = result["draft_answer"]
    state["citations"] = result["citations"]
    _record_node_trace(
        state,
        "synthesize_answer",
        {
            "query_type": state["query_type"],
            "retrieved_doc_count": len(state["retrieved_docs"]),
            "tool_output_count": len(state["tool_outputs"]),
        },
        {"draft_answer": state["draft_answer"], "citations": state["citations"]},
        start,
    )


def verify_grounding_node(
    state: AgentState,
    use_llm: bool = False,
    llm_client: OllamaClient | None = None,
) -> None:
    start = time.perf_counter()
    if use_llm:
        state["verification_result"] = verify_grounding_llm(
            query=state["normalized_query"],
            draft_answer=state["draft_answer"],
            retrieved_docs=state["retrieved_docs"],
            tool_outputs=state["tool_outputs"],
            client=llm_client,
        )
    else:
        state["verification_result"] = verify_grounding(
            query=state["normalized_query"],
            draft_answer=state["draft_answer"],
            retrieved_docs=state["retrieved_docs"],
            tool_outputs=state["tool_outputs"],
        )
    _record_node_trace(
        state,
        "verify_grounding",
        {"draft_answer": state["draft_answer"]},
        state["verification_result"].to_dict(),
        start,
    )
        ```

        **코드 읽기 포인트**
        - `synthesize_answer_node()`와 `verify_grounding_node()`만 `use_llm` 분기를 가진다.
        - trace 구조는 그대로라서 규칙 기반과 LLM 기반 실행을 같은 debugger로 읽을 수 있다.
        - 이 패턴 덕분에 모델 경로를 바꿔도 평가 파이프라인과 failure taxonomy가 깨지지 않는다.

        **결과 해석 가이드**
        - `rule_status`와 `llm_status`가 다르면, LLM 경로가 verifier/fallback 판단까지 바꿨다는 뜻이다.
        - `llm_path`가 `fallback`이면 이 실험은 사실상 규칙 기반 경로를 다시 밟은 것이므로 과해석하면 안 된다.
        - `rule_coverage`와 `llm_coverage`를 같이 보면 자연스러움이 아니라 grounding 관점 차이를 읽을 수 있다.

        **💡 면접 포인트**
        - 기존 workflow contract를 유지한 채 일부 node만 LLM으로 교체하면 회귀 위험을 줄이면서 점진적 확장이 가능하다.
        - `use_llm`를 feature flag처럼 두면 DGX 실험과 CPU-safe 기본 경로를 동시에 유지할 수 있다.

### src 코드 펼침: `workflow.py`의 `use_llm` 분기

```python
def synthesize_answer_node(
    state: AgentState,
    use_llm: bool = False,
    llm_client: OllamaClient | None = None,
) -> None:
    if use_llm:
        result = synthesize_answer_llm(..., client=llm_client)
    else:
        result = synthesize_answer(...)
    state["draft_answer"] = result["draft_answer"]
    state["citations"] = result["citations"]


def verify_grounding_node(
    state: AgentState,
    use_llm: bool = False,
    llm_client: OllamaClient | None = None,
) -> None:
    if use_llm:
        state["verification_result"] = verify_grounding_llm(..., client=llm_client)
    else:
        state["verification_result"] = verify_grounding(...)
```

- LLM을 붙이는 지점은 정확히 두 군데다. `synthesize_answer_node`와 `verify_grounding_node`다.
- classification, planning, retrieval, tool decision 같은 앞단 노드는 여전히 규칙 기반이다. 이 노드들은 빠르고 결정적이며, 데이터가 작을 때는 굳이 LLM을 쓰지 않아도 충분하기 때문이다.
- 다시 말해 이 설계는 "워크플로우 전체를 LLM agent로 치환"한 것이 아니라, 자연어 재구성과 의미 기반 검증이 특히 유리한 지점에만 LLM을 쓴 하이브리드 구조다.

### src 코드 펼침: `run_workflow()`가 `llm_client`를 전달하는 방식

```python
def run_workflow(
    query: str,
    retriever: Any,
    top_k: int = DEFAULT_TOP_K,
    trace_path: Path | None = None,
    use_llm: bool = False,
    llm_client: OllamaClient | None = None,
) -> AgentState:
    state = create_initial_state(query)
    effective_llm_client = llm_client or (OllamaClient() if use_llm else None)

    _execute_workflow_steps(
        state,
        WORKFLOW_STEPS,
        retriever=retriever,
        top_k=top_k,
        use_llm=use_llm,
        llm_client=effective_llm_client,
    )
```

- `llm_client`를 외부에서 주입(injection)할 수 있게 했기 때문에, notebook에서는 같은 workflow를 유지한 채 모델만 바꿔 비교할 수 있다.
- `effective_llm_client = llm_client or (OllamaClient() if use_llm else None)`는 사용성이 좋은 기본값이다. 사용자가 아무 것도 넘기지 않아도 `use_llm=True`만 주면 기본 Ollama 설정으로 바로 붙는다.
- 이 구조 덕분에 10번 notebook에서는 같은 workflow를 `qwen3.5:9b`, `4b`, `2b`로 바꿔가며 공정하게 비교할 수 있다.

### 데이터별 query suite

이제 workflow 비교도 하나의 corpus에서만 보지 않고, `demo_queries`와 `tech_queries`를 각각 5개 query type으로 묶어 양쪽에서 반복 실행한다. 이렇게 해야 "LLM 통합이 작은 장난감 데이터에서는 과한가?"와 "실제 기술 문서에서는 도움이 되는가?"를 같은 notebook 안에서 같이 설명할 수 있다.


In [ ]:
import warnings

import pandas as pd
from IPython.display import display

from src.workflow import run_workflow

query_suites = {
    'demo': demo_queries,
    'tech_docs': tech_queries,
}
retriever_map = {
    'demo': demo_retriever,
    'tech_docs': tech_retriever,
}


def preview(text: str, limit: int = 110) -> str:
    compact = " ".join(text.split())
    return compact if len(compact) <= limit else compact[: limit - 3] + "..."

rows = []
warning_rows = []
for dataset_name, query_map in query_suites.items():
    retriever = retriever_map[dataset_name]
    chunk_count = profiles[dataset_name]['profile']['stats']['chunk_count']
    for expected_type, query in query_map.items():
        rule_state = run_workflow(query, retriever=retriever, use_llm=False)
        with warnings.catch_warnings(record=True) as workflow_warnings:
            warnings.simplefilter('always')
            llm_state = run_workflow(query, retriever=retriever, use_llm=True, llm_client=llm_client)

        rows.append(
            {
                'dataset': dataset_name,
                'chunk_count': chunk_count,
                'expected_type': expected_type,
                'predicted_type': rule_state['query_type'],
                'query': query,
                'rule_status': rule_state['final_status'],
                'llm_status': llm_state['final_status'],
                'rule_coverage': rule_state['verification_result'].coverage_score,
                'llm_coverage': llm_state['verification_result'].coverage_score,
                'llm_path': 'live' if llm_live and not workflow_warnings else 'fallback',
                'rule_answer': preview(rule_state['final_answer']),
                'llm_answer': preview(llm_state['final_answer']),
            }
        )
        warning_rows.append(
            {
                'dataset': dataset_name,
                'query_type': expected_type,
                'warning_count': len(workflow_warnings),
                'warnings': ' | '.join(str(item.message) for item in workflow_warnings) or 'No warning',
            }
        )

workflow_comparison = pd.DataFrame(rows)
status_changes = (
    workflow_comparison.assign(status_changed=workflow_comparison['rule_status'] != workflow_comparison['llm_status'])
    .groupby('dataset', as_index=False)['status_changed']
    .sum()
    .rename(columns={'status_changed': 'status_change_count'})
)
workflow_summary = (
    workflow_comparison.groupby('dataset', as_index=False)
    .agg(
        avg_rule_coverage=('rule_coverage', 'mean'),
        avg_llm_coverage=('llm_coverage', 'mean'),
    )
    .merge(status_changes, on='dataset', how='left')
)

display(workflow_comparison)
display(workflow_summary)
pd.DataFrame(warning_rows)


## 결과 해석: 전체 워크플로우 비교

이 비교표는 "LLM이 더 잘했다"를 단정하기 위한 것이 아니라, 어떤 query type에서 최종 상태와 coverage가 달라졌는지 보는 용도다.

읽을 때는 아래 순서를 권한다.
- 먼저 `predicted_type`이 기대와 맞는지 본다. 분류가 틀리면 LLM 여부와 무관하게 전체 경로가 흔들릴 수 있다.
- 그다음 `rule_status`와 `llm_status`를 본다. 둘이 다르면 verifier/fallback에서 차이가 난 것이다.
- 마지막으로 `rule_answer`, `llm_answer`를 읽는다. 이때 문장 매끈함보다 근거 보수성이 유지됐는지를 우선 본다.

자연스러운 답이 항상 더 좋은 답은 아니다. 운영 시스템에서는 "근거 있는 답 + 실패 시 안전한 거절"이 더 중요하다.

### demo vs 실제 데이터 해석

`demo`에서는 규칙 기반과 LLM 기반의 `final_status`, `coverage` 차이가 생각보다 작을 수 있다. 문서가 짧고 질의도 비교적 직접적이기 때문이다. 반대로 `tech_docs`에서 차이가 더 커진다면, 그것은 모델이 더 똑똑해서라기보다 데이터가 더 복잡해졌기 때문에 합성 능력의 차이가 드러난 것이다.

표를 볼 때는 `status_change_count`와 `avg_llm_coverage - avg_rule_coverage`를 같이 보는 것이 좋다. 작은 코퍼스에서 얻은 결론을 실제 데이터에 그대로 일반화하면 안 된다는 메시지가 여기 담겨 있다.


## 실험: 폴백 동작 확인

**목적**
- 실무에서 LLM 서버가 내려갔을 때도 workflow가 멈추지 않고 규칙 기반 경로로 복귀하는지 확인한다.

**핵심 로직**
- 일부러 잘못된 base URL을 가진 `bad_client`를 만들어 `run_workflow(use_llm=True)`에 넣는다.
- 내부에서 LLM synthesizer/verifier가 경고를 내고, 규칙 기반 경로로 폴백해야 한다.

**주요 파라미터**
- `base_url="http://127.0.0.1:9"`: 실패를 강제로 만드는 잘못된 주소다.
- `fallback_warnings`: 폴백이 실제로 발생했는지 확인하는 경고 목록이다.

**결과 해석 가이드**
- `final_status`가 정상적으로 나오면 서버 장애가 workflow 전체 장애로 번지지 않았다는 뜻이다.
- `warning_count > 0`이면 폴백이 발동했다는 뜻이고, 이때 trace는 여전히 끝까지 남아야 한다.
- 마지막 trace 3행을 보면 LLM failure 후에도 verifier/fallback까지 정상적으로 이어졌는지 읽을 수 있다.

**💡 면접 포인트**
- 운영 환경에서 LLM은 품질 향상 장치이지, 시스템 단일 실패 지점(single point of failure)이 되면 안 된다.
- 폴백이 있으면 서버 장애 시에도 baseline 품질은 유지하고, observability도 그대로 가져갈 수 있다.


In [ ]:
import warnings

import pandas as pd
from IPython.display import display

from src.llm_client import LLMConfig, OllamaClient
from src.trace_debug import display_trace
from src.workflow import run_workflow

bad_client = OllamaClient(LLMConfig(base_url='http://127.0.0.1:9', model=llm_client.config.model))
fallback_query = tech_queries['simple_lookup']

with warnings.catch_warnings(record=True) as fallback_warnings:
    warnings.simplefilter('always')
    fallback_state = run_workflow(
        fallback_query,
        retriever=tech_retriever,
        use_llm=True,
        llm_client=bad_client,
    )

fallback_summary = pd.DataFrame(
    [
        {
            'dataset': 'tech_docs',
            'query': fallback_query,
            'final_status': fallback_state['final_status'],
            'final_answer_preview': ' '.join(fallback_state['final_answer'].split())[:120],
            'trace_steps': len(fallback_state['trace']),
            'warning_count': len(fallback_warnings),
        }
    ]
)
display(fallback_summary)
display(pd.DataFrame({'warning': [str(item.message) for item in fallback_warnings] or ['No warning']}))
display_trace(fallback_state['trace'], render=False).tail(3)


## 핵심 정리

이 노트북을 통해 LLM 통합은 "기존 시스템을 버리고 새 모델 기반으로 갈아탄다"가 아니라, 기존 규칙 기반 workflow 위에 generation과 verification 레이어를 선택적으로 얹는 작업임을 확인했다. `OllamaClient`는 로컬 모델과의 통신을 담당하고, `synthesize_answer_llm()`와 `verify_grounding_llm()`는 각각 자연스러운 답변 생성과 구조화된 grounding 판정을 시도한다. 하지만 두 모듈 모두 규칙 기반 폴백을 내장하고 있어, 서버 장애나 파싱 실패가 곧 시스템 장애로 번지지 않는다.

규칙 기반은 재현성·설명 가능성·안정성에 강하고, LLM 기반은 표현력과 압축 능력에 강하다. 따라서 실무에서는 두 경로를 대체재가 아니라 상보적 경로로 설계하는 편이 안전하다.

**💡 면접 포인트**
- LLM 품질은 높일 수 있지만, 속도·비용·비결정성·서버 의존성이 같이 따라온다. 그래서 fallback이 필수다.
- 프롬프트에서 "evidence only"와 citation 형식을 강제해 grounded answer를 유도할 수 있다.
- verifier의 JSON 응답 파싱 실패까지 대비해야 실제 운영에서 안정적인 LLM 통합이 된다.

데이터 규모에 따라 LLM 도입 효과가 달라진다는 점도 함께 확인했다. `demo`처럼 작은 코퍼스에서는 규칙 기반이 이미 강한 baseline이 될 수 있지만, `tech_docs`처럼 문서 수와 chunk 수가 커지면 LLM의 요약·압축·재구성 능력이 더 큰 가치를 만든다.

**💡 면접 포인트**
- 작은 코퍼스에서는 규칙 기반으로도 충분할 수 있고, 큰 코퍼스에서야 LLM 투자 효과가 뚜렷해진다.
- 그래서 LLM 도입 여부는 모델 자체보다도 데이터 복잡도와 retrieval 난이도와 함께 판단해야 한다.
